# Luồng Chunking Dữ Liệu Bằng Gemini

**Nền tảng:** Project Backend — FastAPI + Celery

---

## 1. Tổng Quan Luồng Xử Lý

Khi người dùng upload một file tài liệu (PDF/DOCX/PPTX), hệ thống sẽ thực hiện chuỗi xử lý sau:

```
Upload File (S3)
    │
    ▼
Tạo DB Record (status=pending)
    │
    ▼
Gọi Background Task (Celery / FastAPI BackgroundTasks)
    │
    ▼
① PARSE — Chuyển file → Markdown (Gemini → Marker → PyMuPDF)
    │
    ▼
② CLEAN — Làm sạch & chuẩn hóa Markdown
    │
    ▼
③ STRUCTURE — Phát hiện cấu trúc chương (Heading Tree)
    │
    ▼
④ CHUNK — Chia nhỏ nội dung theo cấu trúc chương
    │
    ▼
⑤ EMBED — Tạo vector embeddings (sentence-transformers, cache Redis)
    │
    ▼
⑥ UPSERT — Lưu vectors vào Pinecone
    │
    ▼
Cập nhật DB → status=completed
```

**Điểm mấu chốt:** Gemini được dùng ở bước ① (PARSE) để chuyển đổi file PDF thành Markdown. Các bước ②–⑥ dùng các thư viện khác (PyMuPDF, sentence-transformers, Redis, Pinecone).

---

## 2. Điểm Đầu Vào — API Upload

**File:** `backend/app/routers/documents.py`

Endpoint `POST /api/v1/documents/upload` nhận file từ client, lưu lên S3/MinIO, tạo bản ghi DB, rồi trigger background task.

In [ ]:
# Đoạn code chính trong upload_document (routers/documents.py)

service = DocumentService(db)
document = await service.upload_document(
    user_id=current_user.id,
    file_content=content,           # bytes của file
    filename=file.filename or "document",
    course_id=UUID(course_id) if course_id else None,
)

# Trigger xử lý nền (FastAPI BackgroundTasks)
background_tasks.add_task(_process_in_background, str(document.id))

# Phát sự kiện WebSocket để FE biết đang xử lý
await mgr.broadcast(str(document.id), {
    "type": "processing_step",
    "step": "queued",
    "percent": 0,
})

---

## 3. Cấu Hình Gemini

**File:** `backend/app/core/config.py`

Các biến môi trường liên quan đến Gemini:

In [ ]:
# Cấu hình Gemini trong Settings (config.py)

GEMINI_API_KEY  : str = ""          # Một API key đơn
GEMINI_API_KEYS : str = ""          # Nhiều key, phân cách bằng dấu phẩy (ưu tiên dùng)
GEMINI_MODEL    : str = "gemini-2.5-flash"  # Model mặc định
LLM_MODEL_GEMINI: str = "gemini-2.5-flash"  # Alias dùng trong parser

# QWEN_VISION_BASE_URL: URL server Marker (GPU server trên Kaggle)
# Được reuse cho Marker server trong bước fallback ②
QWEN_VISION_BASE_URL: str = ""

---

## 4. Luồng Parse 3 Tầng (Fallback Cascade)

**File:** `backend/app/rag/parser.py`

PDF được xử lý qua 3 mức ưu tiên:

| Mức | Phương pháp | Mô tả |
|------|-------------|--------|
| **① (Cao nhất)** | **Gemini 2.5 Flash** | Gọi Google GenAI SDK, multimodal — hiểu ảnh/charts trong PDF |
| **②** | **Marker Remote Server** | Gọi API tới server Marker chạy trên GPU (Kaggle), đa nền tảng |
| **③ (Thấp nhất)** | **PyMuPDF** | Xử lý local CPU, fallback khi cả hai trên đều fail |

### 4.1. Chiến lược chia chunk của Gemini

Gemini xử lý PDF theo **chunk 10 trang**:

In [ ]:
# Chiến lược chia chunk (parser.py, _parse_pdf_gemini)

import fitz  # PyMuPDF

doc = fitz.open(stream=file_bytes, filetype="pdf")
total_pages = len(doc)
CHUNK_SIZE = 10   # ← Mỗi chunk = 10 trang

for start in range(0, total_pages, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, total_pages)
    # Gọi Gemini cho từng chunk độc lập
    result = await _parse_chunk_gemini(doc, start, end, keys, max_cycles=3)
    chunks_data.append({
        "page_start": start,
        "page_end": end,
        "content": result,
    })

# Kết quả: danh sách chunks → nối bằng "\n\n---\n\n"

### 4.2. Retry logic của Gemini — 3 lớp bảo vệ

Mỗi chunk có 3 lớp retry:

```
Cycle 1…N (max 3)
  ├── Key 0: thử tối đa 3 lần (retry với exponential backoff)
  ├── Key 1: thử tối đa 3 lần
  ├── Key 2: thử tối đa 3 lần
  └── Het key → chờ → Cycle tiếp theo
```

**Chiến lược chờ:**
- Nếu API trả `retryDelay` → chờ đúng thời gian đó × 1.5 (max 300s)
- Nếu không → chờ exponential: `2^cycle * 10s` (20s → 40s → 80s)

In [ ]:
# Retry logic trong _parse_chunk_gemini (parser.py)

for cycle in range(1, max_cycles + 1):
    for key_idx, api_key in enumerate(keys):
        for attempt in range(RETRY_MAX):  # 3 lần thử
            client = genai.Client(api_key=api_key)
            # 1. Upload PDF chunk lên Gemini Files API
            pdf_file = client.files.upload(file=chunk_path)
            # 2. Đợi file xử lý xong
            while pdf_file.state.name == "PROCESSING":
                await asyncio.sleep(3)
                pdf_file = client.files.get(name=pdf_file.name)
            # 3. Gọi Gemini generate_content với prompt
            response = client.models.generate_content(
                model=settings.LLM_MODEL_GEMINI,
                contents=[pdf_file, PROMPT],
            )
            return response.text or ""  # Thành công → trả về Markdown

### 4.3. Prompt Gemini để parse PDF thành Markdown

In [ ]:
PROMPT = (
    "Trích xuất toàn bộ nội dung file này thành format Markdown chuẩn. "
    "Giữ nguyên các heading (# ## ###), công thức toán ($$), bảng. "
    "Nếu có hình ảnh/sơ đồ/bức vẽ trong file, hãy mô tả chi tiết nội dung của chúng trong ngoặc vuông "
    "ví dụ: ![Mô tả chi tiết nội dung hình vẽ: các điểm, nhãn, đường kẻ, ký hiệu]. "
    "KHÔNG dùng placeholder như 'image-1.png' hay 'Image 1' — phải mô tả nội dung thực. "
    "Output PURE MARKDOWN, không thêm giải thích."
)

### 4.4. Xử lý khi một số chunk fail

Nếu Gemini chỉ parse được một phần (một số chunk fail), hệ thống:

1. Gọi **Marker Server** để parse toàn bộ file một lần
2. Trích xuất chỉ các page bị fail từ kết quả Marker
3. Merge kết quả Gemini (thành công) + Marker (bù trừ fail)

In [ ]:
# Luồng fill failed chunks (parser.py, _parse_pdf)

if result:  # Gemini có kết quả
    failed_ranges = result.get("failed_ranges", [])
    if failed_ranges:
        # Gọi Marker cho các page bị fail
        fallback_content = await _fill_failed_chunks(
            file_bytes, failed_ranges, filename
        )
        # Merge Gemini (thành công) + Marker (bù fail)
        merged = _merge_markdown(chunks, fallback_content, failed_ranges)
        result["content"] = merged

### 4.5. Marker Server Fallback

In [ ]:
# Marker Remote Server (parser.py, _parse_pdf_marker_server)

# Health check trước
async with httpx.AsyncClient(timeout=10.0) as client:
    resp = await client.get(f"{marker_url}/health")

# Upload & parse
async with httpx.AsyncClient(timeout=120.0) as client:
    resp = await client.post(
        f"{marker_url}/parse-pdf",
        files={"file": (filename, file_bytes, "application/pdf")},
    )

return {
    "content": data.get("markdown", ""),
    "page_count": data.get("page_count", 0),
}

### 4.6. PyMuPDF Fallback (CPU)

In [ ]:
# PyMuPDF fallback (parser.py, _parse_pdf_pymupdf)

doc = fitz.open(stream=io.BytesIO(file_bytes), filetype="pdf")

# Pre-scan 3 trang đầu để detect font size của heading
for page_num in range(min(3, page_count)):
    blocks = page.get_text("dict", ...).get("blocks", [])
    for block in blocks:
        for bline in block["lines"]:
            for span in bline.get("spans", []):
                size = span.get("size", 0)
                if size >= 10:
                    heading_sizes[size] += 1

# Duyệt từng trang, phân loại heading theo font size
if font_size >= h1_size:  →  # Heading 1
elif font_size >= h2_size:  →  ## Heading 2
elif font_size >= h3_size:  →  ### Heading 3
else:  →  plain text

---

## 5. Document Service — Điều phối toàn bộ Pipeline

**File:** `backend/app/services/document_service.py`

Phương thức `process_document()` là trái tim điều phối toàn bộ RAG pipeline.

In [ ]:
# Luồng xử lý trong DocumentService.process_document()

# Bước 1: PARSE — Gọi Gemini/Marker/PyMuPDF
parse_result = await parse_document(
    file_bytes,
    document.file_type,
    progress_callback=self._make_parse_progress_emit(doc_id_str, mgr),
)
markdown_content = parse_result["content"]
total_pages = parse_result.get("page_count", 0)

# Bước 2: CLEAN — Làm sạch Markdown
cleaned_result = clean_markdown(markdown_content)
markdown_content = cleaned_result.cleaned

# Bước 3: STRUCTURE — Phát hiện cấu trúc chương
heading_tree = detect_heading_tree(markdown_content)
total_chapters = len(heading_tree.get("chapters", []))

# Bước 4: CHUNK — Chia nhỏ theo cấu trúc
chunks = semantic_chunk(markdown_content, heading_tree)
for chunk in chunks:
    chunk["document_id"] = doc_id_str

# Lưu kết quả xử lý vào DB (hiển thị được trên FE kể cả embed fail)
await self.update_processing_result(
    document_id=document_id,
    heading_tree=heading_tree,
    total_chapters=total_chapters,
    total_pages_or_slides=total_pages,
    total_chunks=len(chunks),
)

# Bước 5: EMBED — Tạo vector (local sentence-transformers)
enriched = await embed_chunks(chunks, doc_id_str, redis)

# Bước 6: UPSERT — Lưu vào Pinecone theo chapter
for chapter_id, chapter_chunks in chapter_groups.items():
    await self.vector_store.upsert_chunks(
        document_id=doc_id_str,
        chapter_id=chapter_id,
        chunks=chapter_chunks,
    )

---

## 6. WebSocket Progress — Cập nhật real-time cho FE

Trong suốt quá trình parse, mỗi chunk Gemini hoàn thành → gửi 1 sự kiện WebSocket để FE hiển thị progress bar.

In [ ]:
# progress_callback được gọi trong _parse_pdf_gemini sau mỗi chunk

if progress_callback:
    total_chunks = (total_pages + CHUNK_SIZE - 1) // CHUNK_SIZE
    chunk_idx = start // CHUNK_SIZE
    pct = round((chunk_idx + 1) / total_chunks * 100)
    progress_callback("parse", f"Gemini chunk {chunk_idx + 1}/{total_chunks}", pct)

# FE nhận sự kiện:
# {"type": "processing_step", "step": "parse", "message": "Gemini chunk 1/3", "percent": 33}
# {"type": "processing_step", "step": "clean", "message": "Đang làm sạch...", "percent": 50}
# {"type": "processing_step", "step": "structure", "message": "Đang phát hiện chương...", "percent": 65}
# {"type": "processing_step", "step": "chunk", "message": "Đang chia nhỏ (3 chương)...", "percent": 75}
# {"type": "processing_step", "step": "embed", "message": "Đang tạo vector...", "percent": 88}
# {"type": "processing_step", "step": "done", "message": "Xử lý hoàn tất!", "percent": 100}

---

## 7. Background Task — Celery & FastAPI BackgroundTasks

**File:** `backend/app/tasks/document_task.py`

Có 2 cách trigger xử lý nền:

| Cách | Khi nào dùng |
|------|--------------|
| `FastAPI BackgroundTasks` | Development (mặc định, không cần Celery worker) |
| `Celery process_document_task` | Production với nhiều workers |

Cả 2 đều gọi chung `DocumentService.process_document()`.

In [ ]:
# Celery task (document_task.py)

@celery_app.task(
    bind=True,
    max_retries=3,
    autoretry_for=(Exception,),
    retry_backoff=True,
    retry_backoff_max=300,
)
def process_document_task(self: Task, document_id: str) -> dict:
    return _run_document_task(document_id)

def _run_document_task(document_id: str) -> dict:
    async def _run():
        async with async_session_maker() as db:
            service = DocumentService(db)
            result = await service.process_document(uuid.UUID(document_id))
            # Gửi WebSocket khi hoàn thành
            await manager.broadcast(document_id, SSEvent.processing_step(...))
            return result
    # Handle async/sync bridge (Celery runs sync)
    loop = asyncio.new_event_loop()
    return loop.run_until_complete(_run())

---

## 8. Sơ Đồ Tổng Hợp Toàn Luồng

```
┌─────────────────────────────────────────────────────────────┐
│  ① Upload  (FastAPI router)                                │
│     POST /api/v1/documents/upload                           │
│     - Validate file (PDF/DOCX/PPTX, max 100MB)             │
│     - Upload lên S3/MinIO                                   │
│     - Tạo DB record (pending)                             │
│     - Trigger Background Task                              │
└──────────────────────┬──────────────────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────────────────┐
│  ② Background Processing (Celery / BackgroundTasks)       │
│     DocumentService.process_document(document_id)          │
└───────┬───────────────────────────────────────────────────┘
        │
        ▼
┌─────────────────────────────────────────────────────────────┐
│  ③ PARSE — Gemini Parser (3-level cascade)                 │
│                                                            │
│  PDF ──► Gemini 2.5 Flash                                  │
│         ├─ Split PDF: 10 trang/chunk                      │
│         ├─ Mỗi chunk: 3 cycles × N keys × 3 retries       │
│         ├─ Prompt: extract to Markdown + mô tả ảnh       │
│         ├─ Chunk thành công → giữ kết quả                 │
│         └─ Chunk fail → Marker / PyMuPDF bù trừ          │
│                                                            │
│  DOCX/PPTX ──► python-docx / python-pptx (trực tiếp)     │
└───────┬───────────────────────────────────────────────────┘
        │ markdown_content
        ▼
┌─────────────────────────────────────────────────────────────┐
│  ④ CLEAN — Markdown Cleaner                               │
│     - Chuẩn hóa heading levels                             │
│     - Loại bỏ nhiễu (footer, header, page numbers)       │
│     - Xóa duplicate headings                               │
└───────┬───────────────────────────────────────────────────┘
        │ cleaned_markdown
        ▼
┌─────────────────────────────────────────────────────────────┐
│  ⑤ STRUCTURE — Heading Tree Detection                     │
│     detect_heading_tree(markdown) → heading_tree          │
│     heading_tree = {chapters: [{chapter_id, title,        │
│       sections: [{section_id, title, subsections: []}]}]} │
└───────┬───────────────────────────────────────────────────┘
        │ markdown + heading_tree
        ▼
┌─────────────────────────────────────────────────────────────┐
│  ⑥ CHUNK — Semantic Chunking                               │
│     semantic_chunk(markdown, heading_tree) → chunks[]      │
│     - Chia theo chapter boundaries                         │
│     - Mỗi chunk: text, chapter_id, chunk_id               │
└───────┬───────────────────────────────────────────────────┘
        │ chunks[]
        ▼
┌─────────────────────────────────────────────────────────────┐
│  ⑦ EMBED — Vector Embedding (sentence-transformers)       │
│     embed_chunks(chunks, doc_id, redis) → enriched[]       │
│     - Model: BAAI/bge-m3 (1024 dim, local GPU/CPU)         │
│     - Cache Redis: TTL 7 ngày                              │
│     - Graceful degradation nếu embed fail                  │
└───────┬───────────────────────────────────────────────────┘
        │ enriched_chunks[]
        ▼
┌─────────────────────────────────────────────────────────────┐
│  ⑧ UPSERT — Pinecone Vector Store                          │
│     - Group chunks theo chapter_id                         │
│     - upsert_chunks(doc_id, chapter_id, chapter_chunks)    │
│     - Namespace: {doc_id}_{chapter_id}                      │
└───────┬───────────────────────────────────────────────────┘
        │
        ▼
┌─────────────────────────────────────────────────────────────┐
│  ⑨ Update DB Status                                         │
│     - status = "completed" | "processed" (embed fail)     │
│     - Lưu heading_tree, total_chapters, total_chunks       │
│     - WebSocket: "done" event → FE                        │
└─────────────────────────────────────────────────────────────┘
```

---

## 9. Vai Trò Của Từng Thành Phần

| Thành phần | File | Vai trò trong luồng |
|-----------|------|--------------------|
| **Gemini (GenAI SDK)** | `rag/parser.py` | Parse PDF → Markdown (multimodal, hiểu ảnh) |
| **Marker Server** | `rag/parser.py` | Fallback khi Gemini fail |
| **PyMuPDF** | `rag/parser.py` | Fallback cuối cùng (CPU, không GPU) |
| **DocumentService** | `services/document_service.py` | Điều phối toàn pipeline |
| **semantic_chunk** | `rag/chunker.py` | Chia nhỏ markdown theo cấu trúc chương |
| **embed_chunks** | `rag/embedder.py` | Tạo vector bằng sentence-transformers |
| **Redis** | `core/redis_client.py` | Cache embeddings (tránh tính lại) |
| **Pinecone** | `rag/vector_store.py` | Lưu trữ vectors theo doc_id/chapter |
| **WebSocket** | `websocket/manager.py` | Real-time progress cho FE |
| **Celery** | `tasks/document_task.py` | Xử lý nền (production) |
| **Config** | `core/config.py` | Cấu hình API keys, model, chunk size |

---

## 10. Các Tham Số Quan Trọng

| Tham số | Giá trị mặc định | Ý nghĩa |
|---------|------------------|--------|
| `GEMINI_API_KEYS` | `""` (rỗng) | Danh sách API keys, phân cách bằng `,` |
| `LLM_MODEL_GEMINI` | `gemini-2.5-flash` | Model Gemini dùng để parse |
| `CHUNK_SIZE` (Gemini) | `10 trang` | Số trang mỗi chunk khi gọi Gemini |
| `max_cycles` | `3` | Số lần retry cycle cho mỗi chunk Gemini |
| `RETRY_MAX` | `3` | Số lần retry mỗi API key |
| `RAG_CHUNK_SIZE` | `1200` tokens | Kích thước chunk khi semantic chunk |
| `RAG_CHUNK_OVERLAP` | `200` tokens | Overlap giữa các chunk |
| `ST_EMBEDDING_MODEL` | `BAAI/bge-m3` | Model embedding (local) |
| `ST_EMBEDDING_DIM` | `1024` | Chiều vector embedding |